# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published on: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Spatial coverage: {metadata.spatialCoverage}")
print(f"Temporal coverage: {metadata.temporalCoverage}\n")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.record_sets()
print(f"Found {len(record_sets)} record sets.\n")

for idx, record_set in enumerate(record_sets):
    print(f"[{idx+1}] Record Set Name: {record_set.name}")
    print(f"    @id: {record_set.id}")
    print(f"    Description: {getattr(record_set, 'description', 'No description')}")
    # List fields (by @id)
    if hasattr(record_set, 'fields'):
        print(f"    Fields (@id):")
        for field in record_set.fields:
            print(f"        - {field.id} (name: {field.name}, type: {getattr(field, 'data_type', 'Unknown')})")
    print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. 

**Note:** The record sets and fields are referenced by their `@id` fields, as identified in the overview above.

In [ ]:
# For demonstration, we'll load ALL record sets found above into DataFrames.

# List of record set @ids (update based on previous cell output):
record_set_ids = [r.id for r in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print("Loaded DataFrames for record sets:")
for rsid, df in dataframes.items():
    print(f"- {rsid}: shape={df.shape}")

# Display columns and preview the first record set if available
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"\nColumns for record set '{first_id}':\n", dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# For EDA: select the first record set that contains at least one numeric field
import numpy as np

numeric_field_id = None
group_field_id = None
eda_record_set_id = None
for rsid, df in dataframes.items():
    # Try to detect numeric fields (float/int) by the DataFrame dtypes or by field metadata
    for col in df.columns:
        # Try converting to numeric, ignore errors (will result in NaN for non-numeric)
        try:
            if np.issubdtype(df[col].dropna().apply(type).mode().values[0], (int, float, np.integer, np.floating)):
                numeric_field_id = col
                eda_record_set_id = rsid
                break
            # Try conversion
            pd.to_numeric(df[col], errors='raise')
            numeric_field_id = col
            eda_record_set_id = rsid
            break
        except:
            continue
    # Optionally pick a group field (categorical/string, different from numeric)
    if eda_record_set_id:
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
    if eda_record_set_id and numeric_field_id:
        break

if eda_record_set_id is None or numeric_field_id is None:
    print("No suitable numeric field found for EDA.")
else:
    print(f"Using record set: '{eda_record_set_id}'\nNumeric field: '{numeric_field_id}'\nGroup field: '{group_field_id}' (if available)")
    df = dataframes[eda_record_set_id]
    # Ensure our numeric field is truly numeric (convert if needed)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.5)  # Use median as example threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by group_field_id, if present
    if group_field_id and group_field_id in filtered_df.columns:
        # Only group for numeric columns
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. The following cell shows histograms for the selected numeric field, and bar plots for grouped means if grouped data is available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if eda_record_set_id is not None and numeric_field_id is not None:
    df = dataframes[eda_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        # Show boxplot/grouped means
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and explore a FAIR² dataset described by a Croissant schema using `mlcroissant`. We reviewed record sets and fields by `@id`, performed extraction into DataFrames, filtered and normalized numeric data, and visualized distributions. This process enables reproducible FAIR data science workflows and makes transparent both the data structure and operations performed.

Refer to the [FAIR² dataset page](https://sen.science/doi/10.71728/senscience.y7m0-f273) for additional details and metadata.